In [2]:
!lsof -ti:40006 | xargs kill -9


In [2]:
import os
import yaml
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)
GOOGLE_API_KEY = config['google']['api']
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["SERP_API_KEY"] = config['serp']['api']
os.environ["OPENAI_API_KEY"] = config['nautilus']['api']
os.environ["NRP_KEY"] = config['nautilus']['api']
os.environ["NRP_BASE"] = "https://ellm.nrp-nautilus.io/v1"

In [ ]:
import logging
import traceback
import uuid

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
import gradio as gr
import re
import os
import sys
import spacy
import torch
import pandas as pd
import pickle
import requests
import json
import threading
import nest_asyncio
import uvicorn
import time
from rapidfuzz import fuzz
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertModel
from google.adk.tools.agent_tool import AgentTool
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.a2a.utils.agent_to_a2a import to_a2a

sys.path.append('../utils/')
from mxnet_utils import BERTClassifier, CustomVocab

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nlp = spacy.load("en_core_web_md")
analyzer = SentimentIntensityAnalyzer()

with open("../data/vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

label_map = vocabs["label_map"]
topic_vocab = vocabs["topic_vocab"]
author_vocab = vocabs["author_vocab"]
job_vocab = vocabs["job_vocab"]
location_vocab = vocabs["location_vocab"]
affiliation_vocab = vocabs["affiliation_vocab"]

conservative_bigrams = pd.read_csv('../data/top_conservative_bigrams.csv')['bigram'].tolist()
liberal_bigrams = pd.read_csv('../data/top_liberal_bigrams.csv')['bigram'].tolist()

spam_tokenizer = AutoTokenizer.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")
spam_model = AutoModelForSequenceClassification.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_base = BertModel.from_pretrained("bert-base-uncased")

net_best2 = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=32,
    author_dropout=0.1,
    author_mlp_layers=2,
    author_mlp_hidden=192,
    history_dropout=0.1,
    history_mlp_layers=2,
    history_mlp_hidden=192,
)
net_best2.load_state_dict(torch.load("../checkpoints/best.pth", map_location=device))
net_best2.to(device)
net_best2.eval()

def func_political_bias(text: str) -> str:
    statistic_types = {"CARDINAL", "PERCENT", "MONEY", "QUANTITY"}
    doc = nlp(str(text))
    stat_count = sum(ent.label_ in statistic_types for ent in doc.ents)

    def count_matches(stmt, bigram_list):
        words = [w.text.lower() for w in nlp(str(stmt))]
        if len(words) < 2:
            return 0
        bigrams = ["".join(words[i:i+2]) for i in range(len(words)-1)]
        matches = 0
        for bg in bigrams:
            for check in bigram_list:
                if fuzz.ratio(bg, check) >= 70:
                    matches += 1
                    break
        return matches

    return json.dumps({
        "stat_density": stat_count,
        "conservative_talking_points": count_matches(text, conservative_bigrams),
        "liberal_talking_points": count_matches(text, liberal_bigrams),
    })

def func_sensationalism(text: str) -> str:
    score = analyzer.polarity_scores(str(text))["compound"]
    return json.dumps({"emotional_intensity": abs(score), "polarity": score})

def func_spam(text: str) -> str:
    inputs = spam_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = spam_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
    return json.dumps({"spam_probability": probs[0,1].item()})

def func_BERT(text: str) -> str:
    enc = bert_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    input_ids = enc["input_ids"]
    token_types = enc.get("token_type_ids", torch.zeros_like(input_ids)).to(device)
    mask = enc["attention_mask"]

    with torch.no_grad():
        outputs = net_best2(
            input_ids,
            token_types,
            mask,
            torch.zeros(1, len(topic_vocab)).to(device),
            torch.tensor([0]).to(device),
            torch.tensor([0]).to(device),
            torch.tensor([0]).to(device),
            torch.tensor([0]).to(device),
            torch.zeros(1, len(label_map)).to(device),
        )
        probs = torch.softmax(outputs, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        rev = {v:k for k,v in label_map.items()}

    return json.dumps({
        "model_prediction": rev[pred],
        "confidence": probs[0,pred].item(),
        "class_probabilities": {rev[i]: probs[0,i].item() for i in range(len(label_map))}
    })

def func_web_search(text: str) -> str:
    url = "https://serpapi.com/search"
    params = {"q": text, "api_key": os.environ.get("SERP_API_KEY"), "engine": "google", "num": 4}
    try:
        res = requests.get(url, params=params).json()
        evidence = []
        if "answer_box" in res:
            evidence.append(res["answer_box"].get("answer") or res["answer_box"].get("snippet"))
        for item in res.get("organic_results", []):
            evidence.append(item.get("snippet"))
        return "\n".join(evidence) if evidence else "No live evidence found."
    except:
        return "Search Error"

worker_llm = LiteLlm(model="gemini/gemini-3-flash-preview", api_key=os.environ.get("GOOGLE_API_KEY"))
manager_llm = LiteLlm(model="gemini/gemini-3-pro-preview", api_key=os.environ.get("GOOGLE_API_KEY"))

worker_instruction = "Call your tool immediately with the text you receive. Return only the tool output."

bias_agent = LlmAgent(name="Political_Bias_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_political_bias])
sensational_agent = LlmAgent(name="Sensationalism_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_sensationalism])
spam_agent = LlmAgent(name="Spam_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_spam])
bert_agent = LlmAgent(name="BERT_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_BERT])
search_agent = LlmAgent(name="Web_Search_Agent", model=worker_llm, instruction=worker_instruction, tools=[func_web_search])

manager_prompt = """
CRITICAL:
Your final message MUST be ONLY valid JSON.
Do not include markdown.
Do not include explanations outside JSON.
Do not include code fences.
You are the Factuality Root Manager. Your job is to generate a final fact-checking JSON report.

### CRITICAL: MANDATORY DATA GATHERING
You MUST consult your sub-agents to gather data BEFORE generating your final response to the user. Do not stop or reply to the user until you have collected sufficient information from AT LEAST ONE of the following agents:

1. Call 'BERT_Agent' to get truthfulness probabilities.
2. Call 'Political_Bias_Agent' to check for stats and partisan framing.
3. Call 'Sensationalism_Agent' to get the emotional intensity score.
4. Call 'Spam_Agent' to check for bot-like characteristics.
5. Call 'Web_Search_Agent' to look for live evidence using keywords.
6. Call 'RAG_Agent' to check internal archives.

Wait for each agent to return its data, keep it in your internal memory, and immediately call the next agent on the list.

### FINAL SYNTHESIS
ONLY AFTER you have received data from all 6 agents, synthesize the results and output the final report.

[ORIGINAL FACTUALITY INSTRUCTIONS]
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.
Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

VECTOR DESCRIPTION:
0-5: Probabilities for truthfulness classes (BERT_Agent).
7: Count of numeric/statistical entities.
8: Count of conservative bigram matches.
9: Count of liberal bigram matches.
10: Emotional intensity score (Sensationalism_Agent).
11: Spam likelihood score (Spam_Agent).

ANTI-BIAS CONSTRAINT:
- Treat predictive model scores only as auxiliary context. Rely on TEXTUAL EVIDENCE for final verdicts.

FACTUALITY FACTORS:
1. AUTHENTICITY (1–10): Verifiable details, sources, timestamps.
2. SENSATIONALISM (1–10): Density of hyperbole/drama.
3. POLITICAL BIAS (0–10 + tag): Partisan framing/selective omission.
4. SPAM (1–10): Bot-like content characteristics.
5. CONFIRMATION BIAS (1–10): Cherry-picked evidence.
6. SHORT-TERM UTILITY (1–10): Clickbait or monetization cues.

OUTPUT FORMAT (STRICT JSON):
{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "...",
  "factor_scores": [
    {"factor": "Authenticity", "score": 1-10, "reasoning": "..."},
    ...
  ]
}
"""

manager_agent = LlmAgent(
    name="FactCheck_Manager",
    model=manager_llm,
    instruction=manager_prompt,
    tools=[
        AgentTool(agent=bert_agent),
        AgentTool(agent=bias_agent),
        AgentTool(agent=sensational_agent),
        AgentTool(agent=spam_agent),
        AgentTool(agent=search_agent),
    ],
)

nest_asyncio.apply()
a2a_app = to_a2a(manager_agent)

def run_server():
    uvicorn.run(a2a_app, host="0.0.0.0", port=40004, log_level="error")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

def call_agent_api(text: str) -> str:
    base_url = "http://localhost:40004/"

    send_payload = {
        "jsonrpc": "2.0",
        "method": "message/send",
        "id": 1,
        "params": {
            "message": {
                "messageId": str(uuid.uuid4()),  # only messageId, NO taskId
                "role": "user",
                "parts": [{"kind": "text", "text": text}]
            }
        }
    }

    try:
        logger.info("Sending request")
        r = requests.post(base_url, json=send_payload, timeout=180)
        logger.info(f"Status: {r.status_code}")
        r.raise_for_status()

        j = r.json()
        logger.info(f"Raw response:\n{json.dumps(j, indent=2)}")

        if "error" in j:
            logger.error(f"JSON-RPC error: {j['error']}")
            return ""

        return extract_text_from_a2a_response_v03(j)

    except requests.exceptions.Timeout:
        logger.error("Request timed out.")
        return ""
    except Exception as e:
        logger.error(f"call_agent_api failed:\n{traceback.format_exc()}")
        return ""

def extract_text_from_a2a_response_v03(j: dict) -> str:
    """Parse A2A v0.3.0 response envelope."""
    result = j.get("result", {})

    for part in result.get("parts", []):
        if part.get("kind") == "text" and part.get("text"):
            return part["text"]

    for artifact in result.get("artifacts", []):
        for part in artifact.get("parts", []):
            if part.get("kind") == "text" and part.get("text"):
                return part["text"]

    status_msg = result.get("status", {}).get("message", {})
    for part in status_msg.get("parts", []):
        if part.get("kind") == "text" and part.get("text"):
            return part["text"]

    logger.warning(f"Could not extract text. Full result:\n{json.dumps(result, indent=2)}")
    return ""
def run_full_pipeline_agentic(article_text, progress=gr.Progress()):
    if not article_text.strip():
        return "Error", [], "Empty input", {}

    resp = call_agent_api(article_text)
    if not resp:
        return "Error", [], "Agent returned empty response", {}

    m = re.search(r"\{.*\}", resp, re.DOTALL)
    if not m:
        return "Error", [], resp, {}

    data = json.loads(m.group(0))

    veracity = data.get("veracity_label", "Unknown")
    explanation = data.get("explanation_text", "")
    scores = data.get("factor_scores", [])
    df = [[s["factor"], s["score"], s["reasoning"]] for s in scores]

    return veracity, df, explanation, {"status": "ok"}

with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:
    gr.Markdown("# Agentic News Fact-Analysis")

    with gr.Row():
        with gr.Column():
            article_input = gr.Textbox(lines=15, label="Input Article")
            btn = gr.Button("Run Analysis")

        with gr.Column():
            verdict = gr.Label(label="Manager Verdict")
            summary = gr.Markdown()
            table = gr.Dataframe(headers=["Factor", "Score", "Reasoning"])
            meta = gr.JSON()

    btn.click(run_full_pipeline_agentic, article_input, [verdict, table, summary, meta])

demo.queue()
demo.launch()

C:\Users\Chris Mo\AppData\Local\Temp\ipykernel_34308\4027402146.py:152: UserWarning: [GEMINI_VIA_LITELLM] gemini/gemini-3-flash-preview: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='gemini/gemini-3-flash-preview') with Gemini(model='gemini-3-flash-preview'). Set ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS=true to suppress this warning.
  worker_llm = LiteLlm(model="gemini/gemini-3-flash-preview", api_key=os.environ.get("GOOGLE_API_KEY"))
C:\Users\Chris Mo\AppData\Local\Temp\ipykernel_34308\4027402146.py:153: UserWarning: [GEMINI_VIA_LITELLM] gemini/gemini-3-pro-preview: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='gemini/gemini-3-pro-preview') with Gemini(model='gemini-3-pro-preview'). Set ADK_

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


2026-02-25 14:43:51,976 - INFO - Sending request
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\adk\a2a\utils\agent_to_a2a.py:120: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service=InMemoryCredentialService(),
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\adk\auth\credential_service\in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\adk\a2a\executor\a2a_agent_executor.py:190: UserWarning: [EXPERIMENTAL] convert_a2a_request_to_agent_run_request: ADK Implementation for A2A suppo

14:43:54 - LiteLLM:INFO: utils.py:3748 - 
LiteLLM completion() model= gemini-3-pro-preview; provider = gemini


2026-02-25 14:43:54,041 - INFO - 
LiteLLM completion() model= gemini-3-pro-preview; provider = gemini


...

2026-02-25 14:44:16,218 - INFO - Closing runner...
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 7: Expected `Message` - serialized value may not be as expected [input_value=Message(content='Trump\'s...RhZ82dnkLk9wkAixsjIh']}), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...hZ82dnkLk9wkAixsjIh']})), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
2026-02-25 14:44:16,219 - INFO - 
LiteLLM completion() model= gemini-3-flash-preview; provider = gemini
2026-02-25 14:44:16,222 - INFO - Runner closed.
2026-02-25 14:44:16,227 - INFO - Closing runner...
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:


14:44:20 - LiteLLM:INFO: utils.py:3748 - 
LiteLLM completion() model= gemini-3-flash-preview; provider = gemini


2026-02-25 14:44:20,458 - INFO - 
LiteLLM completion() model= gemini-3-flash-preview; provider = gemini
2026-02-25 14:44:27,534 - INFO - Closing runner...
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 7: Expected `Message` - serialized value may not be as expected [input_value=Message(content='The quot...m+NPvIrKkBO2/JzZevkl']}), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...+NPvIrKkBO2/JzZevkl']})), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
2026-02-25 14:44:27,536 - INFO - Runner closed.


14:44:27 - LiteLLM:INFO: utils.py:3748 - 
LiteLLM completion() model= gemini-3-pro-preview; provider = gemini


2026-02-25 14:44:27,543 - INFO - 
LiteLLM completion() model= gemini-3-pro-preview; provider = gemini
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 7: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{\n  "ve...pcQKAM+3XIADmGswFv6S']}), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...cQKAM+3XIADmGswFv6S']})), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
2026-02-25 14:44:37,052 - INFO - Status: 200
2026-02-25 14:44:37,053 - INFO - Raw response:
{
  "id": 1,
  "jsonrpc": "2.0",
  "result": {
    "artifacts": [
      {
        "artifactId": "e17e6836-a843-40c9-8cb1-c8173253f6bf",
        "parts": [
          {
            "kind": "text",
      